In [1]:
#premise_pathways
import bw2data
import bw2io as bi
from datapackage import Package
import premise as pr
from por_supplier_linking import configure_por_export_support, attach_por_supplier_linking
configure_por_export_support()
import bw2data as bd
import pycountry
import re
import pandas as pd

# Now import shared variables
from config import *
from private_keys import *

#Standard vars
bw2data.projects.set_current(PROJECT_NAME)
pr.__version__,PROJECT_NAME

11:49:27+0100 [warning  ] Can't import `SimaProBlockCSVImporter` - please install `bw2io` with `pip install bw2io[multifunctional]` or install `multifunctional` and `bw_simapro_csv` manually.
Applied premise external-market compatibility fix for empty uncertainty bounds.
Applied export guard for scenarios with validated repo-managed PoR supplier links.
Installed repo Pathways mapping compatibility guard for metals diagnostics.
Enabled guarded lithium source-inventory duplicate review; quantities and uncertainty are unchanged.
Installed exact inventory supplier compatibility hooks for configured additional inventories.


((2, 5, 5), 'por')

### 1. Select scope and export years
The presets supply defaults only. Scenario dictionaries, native premise constructors, update calls, supplier hooks and exports remain visible below. Select the same scope in notebook 2.


In [2]:
# Scope settings only; the native build/export calls remain below.
from config import ASSESSMENT_SCOPES
ASSESSMENT_SCOPE = "por_polymers"  # or "por_polymers", "forecast_steel", "european_steel"
scope = ASSESSMENT_SCOPES[ASSESSMENT_SCOPE]
create_db_package = True
create_dbs = True

PACKAGE_YEARS = list(scope["years"])
#PACKAGE_YEARS = [2050]
BRIGHTWAY_YEARS = [2050]             # independent, explicit Brightway inspection years
BRIGHTWAY_SCENARIO_INDICES = None    # None exports all cases; [0] selects the first
PATHWAYS_PACKAGE_BASENAME = scope["package"]
bau_iam_scenario = "SSP2-PkBudg1000"

FORECAST_STEEL_SCENARIOS = None  # None uses all cases in the separate FORECAST steel workbook

CONVERT_OPEN_PROM_INPUT = False      # input preparation, not needed for every export
for label, years in [("Pathways", PACKAGE_YEARS), ("Brightway", BRIGHTWAY_YEARS)]:
    if not years or len(set(years)) != len(years) or any(type(y) is not int for y in years):
        raise ValueError(f"Select unique integer years for {label}.")


### Shared inventory for the PoR scope
For `por_polymers`, save and sync the shared workbook before continuing. The existing snapshot and hash checks remain active. The steel scope does not refresh this workbook.


In [3]:
import importlib
import config as model_config
importlib.reload(model_config)
from config import stage_itom_inventory, verify_itom_inventory_snapshot

itom_inventory_snapshot = None
if create_dbs or create_db_package:
    if ASSESSMENT_SCOPE == "por_polymers":
        itom_inventory_snapshot = stage_itom_inventory()
        print("ITOM inventory source:", itom_inventory_snapshot["source_path"])
        print("Frozen workbook SHA256:", itom_inventory_snapshot["sha256"])
        print("Inventory snapshot record:", itom_inventory_snapshot["manifest_path"])
    else:
        print("Steel scope: no shared PoR workbook is staged.")
else:
    print("Both export routes disabled; inventory snapshot was not refreshed.")

ITOM inventory source: C:\Users\terlouw_t\OneDrive - Paul Scherrer Institut\lci-itom_por.xlsx
Frozen workbook SHA256: 3e0f6290815b3f77ab9ba039048318ad6f2ad32786d0816ba5c81d0ccd48d6e9
Inventory snapshot record: C:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\results\input_snapshots\3e0f6290815b3f77ab9ba039048318ad6f2ad32786d0816ba5c81d0ccd48d6e9\20261107T104928851794Z_0911135f.json


### 1.1 Decide on the scenarios to be included for this exercise:

Choose one non-ferrous recycling case below for **both Brightway and Pathways exports**. Share files are prepared automatically in `primary_vs_secondary_metals/scenario_data`; non-baseline outputs receive a case suffix. **Steel remains endogenous to IAM/EDM-I.**

#### Non-ferrous metals recycling scenarios

Set `NONFERROUS_METALS_SCENARIO` to **one exact column heading below**. Values are recycled **input shares**, not end-of-life recycling rates. For the three minimum cases, the percentages shown are **2050 floors**, not fixed shares for every year.

| Product / application | `baseline` | `reference` | `conservative` | `moderate_recycling` | `high_recycling` | `primary_only` |
| --- | ---: | ---: | ---: | ---: | ---: | ---: |
| **How applied** | Native premise | Fixed 2020 share | Minimum floor | Minimum floor | Minimum floor | Zero approved secondary |
| Aluminium, cast alloy | Native | 68.82% | 45% | 55% | 65% | 0% |
| Copper, cathode | Native | 20% | 25% | 35% | 45% | 0% |
| Nickel, class 1 | Native | 0.203% | 15% | 22% | 30% | 0% |
| Cobalt | Native | 0.572% | 20% | 30% | 40% | 0% |
| Lead | Native | 54.92% | 65% | 77% | 85% | 0% |
| Lithium carbonate (ordinary, GLO) | Native | 0.0840% | 15% | 22.28% | 37.31% | 0% |
| Lithium carbonate, battery grade (World only) | Native | 0.0840% | 15% | 22.28% | 37.31% | 0% |
| Gold | Native | 0.443% | 27.7% | 37.7% | 47.7% | 0% |
| Silver | Native | 8.464% | 18.9% | 28.9% | 38.9% | 0% |
| Platinum | Native | 4.460% | 21.8% | 31.8% | 41.8% | 0% |
| Palladium | Native | 15.549% | 32.2% | 42.2% | 52.2% | 0% |

- **Minimum cases:** applied secondary share = `max(native_share, interpolated_floor(year))`. A higher native share is never reduced. Primary/non-secondary remainder = `1 - applied_secondary_share`.
- **Years:** non-precious products, including both carbonate rules, interpolate linearly from 2020 to 2050. Precious metals pass through shared **2025 anchors** (gold 27.7%, silver 18.9%, platinum 21.8%, palladium 32.2%) before their 2050 floors. Endpoints clamp outside the covered years. The moderate/high precious-metal increments of **10/20 percentage points are analyst sensitivities, not published forecasts**.
- **Names:** `baseline` means version-native premise, not the file `primary_secondary_split_baseline.yaml`; that file supplies `moderate_recycling`. `reference` fixes shares only; future process/background inventories still evolve.
- **Evidence refresh (23 October 2026):** aluminium/copper floors are analyst sensitivities; nickel/cobalt are rounded battery-derived secondary/global-total-demand proxies, not universal grade shares. Lithium's 15% is an analyst lower-collection case; 22.28/37.31% are source-checked approximate elemental-Li ratios, with low-confidence carbonate-market applicability.
- **Scope:** `World`/`GLO`, except ordinary lithium carbonate (`GLO`) and battery-grade carbonate (`World` only). Ordinary hydroxide benefits through its existing carbonate input, with no additional override or credit. Battery-grade GLO carbonate/hydroxide remain native. Existing recovery recipes and burdens remain, with documented grade/scrap-mix proxies. Silver's Parkes stream stays in the non-secondary remainder. Steel and unsupported products remain native.
- **Lithium limits:** global elemental-Li fractions are representative-compound sensitivities, not compound-grade forecasts. The existing RELICS Lithium-only definition omits `Spodumene`; do not interpret its change as complete lithium-extraction accounting.
- **Deferred evidence:** graphite and magnet-REE candidate percentages do not enable market overrides. Only Nd/Pr/Dy/Tb receive magnet-REE candidate curves; the mapping's `reference_only` scope retains the thirteen other rare-earth source curves. All rare-earth runtime markets remain native.

Details: [policy and scenario-file mapping](docs/metals_scenario_policy.md#available-non-ferrous-cases), [evidence and assumptions](docs/metals_literature_review.md), and the metals workbook's `Scenario assumptions`/`Evidence` sheets.

In [4]:
# One policy selection for both CL/OCE and both Brightway/Pathways exports.
from pathlib import Path
from config import ROOT, DATAPACKAGE_DIR
import metals_scenario_policy as metal_policy_module
import primary_vs_secondary_metals.share_files as metal_share_files
NONFERROUS_METALS_SCENARIO = "baseline" #  #normal baseline, reference, conservative, moderate_recycling, high_recycling, primary_only
#NONFERROUS_METALS_SCENARIO = "high_recycling" #  #normal baseline, reference, conservative, moderate_recycling, high_recycling, primary_only

In [5]:
METALS_SHARE_SOURCE = "yaml"  # YAML + evidence/mapping CSV inputs; use xlsx to read the Shares workbook instead
FORCE_METALS_SHARE_REGENERATION = False  # True only to deliberately replace conflicting manual scenario edits
METALS_POLICY_FILE = Path(metal_policy_module.__file__).resolve().parent / "configuration_file/metals_scenario_policy.yaml"


def load_selected_metal_steel_policy():
    share_preparation = metal_share_files.prepare_policy_share_data(
        METALS_POLICY_FILE, source=METALS_SHARE_SOURCE, force=FORCE_METALS_SHARE_REGENERATION,
    )
    print("Metals share data:", share_preparation.get("status"),
          "; products:", share_preparation.get("product_count"))
    return metal_policy_module.MetalSteelPolicy.from_file(
        METALS_POLICY_FILE, nonferrous_case=NONFERROUS_METALS_SCENARIO,
    )

# Validate the selection before constructing expensive scenario builders.
selected_metal_steel_policy = load_selected_metal_steel_policy()
print("Selected non-ferrous metals:", NONFERROUS_METALS_SCENARIO, "; steel: selected IAM/external modules")


# Prepare only the selected steel source. The raw workbook stays unchanged.
if ASSESSMENT_SCOPE == "forecast_steel":
    from runpy import run_path
    forecast_steel_converter = run_path(str(ROOT / "0_convert_FORECAST_steel_to_IAMC.py"))
    print("FORECAST steel input:", forecast_steel_converter["transform_forecast_steel"]())

# The external modules and their order stay explicit here.
sps_op = Package(str(DATAPACKAGE_DIR / "datapackage_open_prom.json"), base_path=str(ROOT))
sps_edm_i = Package(str(DATAPACKAGE_DIR / "datapackage_edm_i.json"), base_path=str(ROOT))
sps_forecast = Package(str(DATAPACKAGE_DIR / "datapackage_forecast.json"), base_path=str(ROOT))
sps_itom_por = Package(str(DATAPACKAGE_DIR / "datapackage_itom_por.json"), base_path=str(ROOT))
sps_itom = Package(str(DATAPACKAGE_DIR / "datapackage_itom.json"), base_path=str(ROOT))

if ASSESSMENT_SCOPE == "por_polymers":
    scenarios = [
        {"model": "remind", "year": 2050, "pathway": bau_iam_scenario,
         "external scenarios": [
             {"scenario": "fc_nz", "data": sps_forecast},
             {"scenario": "OCE", "data": sps_itom_por},
         ]},
        {"model": "remind", "year": 2050, "pathway": bau_iam_scenario,
         "external scenarios": [
             {"scenario": "fc_nz", "data": sps_forecast},
             {"scenario": "CL", "data": sps_itom_por},
         ]},
    ]
elif ASSESSMENT_SCOPE == "forecast_steel":
    sps_forecast_steel = Package(str(DATAPACKAGE_DIR / "datapackage_forecast_steel.json"), base_path=str(ROOT))
    available_forecast_cases = sps_forecast_steel.descriptor["scenarios"]
    # Accept original source IDs ("1", "2") or exported text scenario names.
    forecast_source_ids = sps_forecast_steel.descriptor.get("source_scenarios", {})
    forecast_cases = available_forecast_cases if FORECAST_STEEL_SCENARIOS is None else [
        forecast_source_ids.get(str(case), str(case)) for case in FORECAST_STEEL_SCENARIOS
    ]
    if not forecast_cases or len(set(forecast_cases)) != len(forecast_cases) or not set(forecast_cases) <= set(available_forecast_cases):
        raise ValueError(f"Select unique FORECAST steel cases from {available_forecast_cases}.")
    scenarios = [
        {"model": "remind", "year": 2050, "pathway": bau_iam_scenario,
         "external scenarios": [
             {"scenario": "NDC-EI", "data": sps_op},
             {"scenario": case, "data": sps_forecast_steel}]}
        for case in forecast_cases
    ]
elif ASSESSMENT_SCOPE == "european_steel":
    scenarios = [
        {"model": "remind", "year": 2050, "pathway": bau_iam_scenario,
         "external scenarios": [
             {"scenario": "NDC-EI", "data": sps_op},
             {"scenario": "ITOM_Steel_Test-scenario", "data": sps_itom},
         ]},
    ]
else:
    raise ValueError(f"Define scenario dictionaries for {ASSESSMENT_SCOPE!r} in this cell.")

display(pd.DataFrame([
    {"case": i, "background": sc["pathway"],
     "external modules in order": " → ".join(entry["scenario"] for entry in sc["external scenarios"]),
     "Pathways years": PACKAGE_YEARS, "Brightway years": BRIGHTWAY_YEARS}
    for i, sc in enumerate(scenarios)
]))


Metals share data: reused ; products: 104
Selected non-ferrous metals: baseline ; steel: selected IAM/external modules


,case,background,external modules in order,Pathways years,Brightway years
0,0,SSP2-PkBudg1000,fc_nz → OCE,"[2025, 2030, 2035, 2040, 2045, 2050]",[2050]
1,1,SSP2-PkBudg1000,fc_nz → CL,"[2025, 2030, 2035, 2040, 2045, 2050]",[2050]


In [6]:
# we need to convert ISO3 names to ISO2 for the PoR data using Open-prom, which is in ISO2. We can do this with pycountry.

In [7]:
if CONVERT_OPEN_PROM_INPUT:
    # Read CSV
    df = pd.read_csv("scenario_data/scenario_data_open_prom_init.csv")

    def iso3_to_iso2(code):
        try:
            return pycountry.countries.get(alpha_3=code).alpha_2
        except:
            return code  # keep original if not found

    df["region"] = df["region"].apply(iso3_to_iso2)

    # Save
    df.to_csv("scenario_data\scenario_data_open_prom.csv", index=False)

<>:14: SyntaxWarning: invalid escape sequence '\s'
<>:14: SyntaxWarning: invalid escape sequence '\s'
C:\Users\terlouw_t\AppData\Local\Temp\ipykernel_30164\2757247816.py:14: SyntaxWarning: invalid escape sequence '\s'
  df.to_csv("scenario_data\scenario_data_open_prom.csv", index=False)


In [8]:
# Create dbs if not existing yet
if DB_NAME_INIT not in bi.databases:
    bi.import_ecoinvent_release("3.12", "cutoff", USER_NAME, USER_PW)
bi.databases

Databases dictionary with 9 object(s):
	ecoinvent-3.12-biosphere
	ecoinvent-3.12-cutoff
	ecoinvent_312_reference
	remind_2050_SSP2-PkBudg1000_NDC-EI_ITOM_Steel_Test-scenario
	remind_2050_SSP2-PkBudg1000_NDC-EI_forecast_steel_1
	remind_2050_SSP2-PkBudg1000_NDC-EI_forecast_steel_2
	remind_2050_SSP2-PkBudg1000_fc_nz_CL
	remind_2050_SSP2-PkBudg1000_fc_nz_OCE
	remind_2050_SSP2-PkBudg1000_fc_nz_OCE__metals-high_recycling

In [9]:
for db in list(bi.databases):
    if db == 'ecoinvent_312_reference':
        pass
        #del bd.databases['ecoinvent_312_reference']
    if 'remind_' in db:
        pass
        #print(f"Deleting {db}...")
        #del bd.databases[db]
list(bi.databases)

['ecoinvent-3.12-biosphere',
 'ecoinvent-3.12-cutoff',
 'remind_2050_SSP2-PkBudg1000_fc_nz_OCE__metals-high_recycling',
 'ecoinvent_312_reference',
 'remind_2050_SSP2-PkBudg1000_NDC-EI_ITOM_Steel_Test-scenario',
 'remind_2050_SSP2-PkBudg1000_NDC-EI_forecast_steel_1',
 'remind_2050_SSP2-PkBudg1000_NDC-EI_forecast_steel_2',
 'remind_2050_SSP2-PkBudg1000_fc_nz_OCE',
 'remind_2050_SSP2-PkBudg1000_fc_nz_CL']

In [10]:
# We need to make an additional database with ecoinvent to import LCIs
def generate_reference_database():
    """
    Build and validate a reference database before replacing its exact target.

    Native validation and the guarded source-inventory compatibility check run
    before deletion. Unrelated databases, including similarly named ones, remain.

    Returns:
    None

    Example:
    >>> generate_reference_database()
    """

    from premise_compat import preflight_reference_database_export

    if NAME_REF_DB == DB_NAME_INIT:
        raise ValueError("The reference export target must differ from the source database.")
    bd.projects.set_current(PROJECT_NAME)
    pr.clear_cache()
    # Create a new reference database using NewDatabase
    ndb = pr.NewDatabase(
        scenarios=[{"model": "remind", "pathway": "SSP2-NPi", "year": "2025",
                    "exclude": ["update_electricity", "update_cement", "update_steel", "update_dac",
                                "update_fuels", "update_emissions"]}],
        source_db=DB_NAME_INIT,
        source_version=EI_VERSION,
        key=KEY_PREMISE,
        biosphere_name=BIOSPHERE_DB,
    )

    # Check the replacement before deleting the existing reference database.
    preflight_reference_database_export(ndb)
    if NAME_REF_DB in bd.databases:
        print(f"Replacing {NAME_REF_DB} after successful reference preflight")
        del bd.databases[NAME_REF_DB]

    # Export validation remains active when writing the new reference.
    ndb.write_db_to_brightway(name=NAME_REF_DB)

if NAME_REF_DB not in bi.databases:
    generate_reference_database()

### 1.4 if we want to export it as a brightway database, we can do this as follows:

In [11]:
def clean(s: str) -> str:
    """Replace spaces with - and ensure it's filename safe."""
    return re.sub(r"\s+", "-", s)

scenario_names = []

for sc in scenarios:
    parts = [sc["model"], str(sc["year"]), sc["pathway"]]
    external = [clean(item["scenario"]) for item in sc["external scenarios"]]
    parts.extend(external)
    scenario_names.append("_".join(parts))
scenario_names

['remind_2050_SSP2-PkBudg1000_fc_nz_OCE',
 'remind_2050_SSP2-PkBudg1000_fc_nz_CL']

### Syngas methanol, MTO/MTA and ethylene-oxide carbon origins before export

For PoR polymers, both export routes below check the imported stage-based syngas-methanol/MTO/MTA recipes and regionalized ethylene oxide after supplier linking. The linked methanol/CO2 make-up supplies determine the carbon-origin fractions for each case and year. The ethylene-oxide correction follows the linked ethylene market, including its MTO/MTA carbon origins. Existing process CO2 is split between fossil and non-fossil biosphere flows; DAC and biogenic carbon remain separate in the audit. Utilities, total emitted CO2 and allocation remain unchanged.

The adapter checks the reconstructed joint-process balance. Current positive gaps of about 1–3% are reported, not filled artificially. Syngas methanol currently has an 8.1% gap and is held unchanged; its output origin remains unknown for downstream MTO/MTA. Unresolved origins retain the fossil label; unreviewed legacy recipes are flagged. See [workflow and assumptions](docs/por_carbon_emissions.md). Rebuild the package and recalculate notebook 2 to use the correction.

```mermaid
flowchart LR
    A[Import inventories] --> B[Create yearly markets and link suppliers]
    B --> C[Check balances and trace methanol, syngas and ethylene carbon]
    C --> D[Split process CO2 into fossil and non-fossil flows]
    D --> E[Validate, record audit and export]
```


In [12]:
if create_dbs:
    import importlib
    import premise_compat as por_compat
    import por_supplier_linking as por_linking
    import metals_scenario_policy as metal_policy_module
    import primary_vs_secondary_metals.share_files as metal_share_files
    importlib.reload(metal_policy_module)
    importlib.reload(metal_share_files)
    importlib.reload(por_compat)
    importlib.reload(por_linking)
    selected_metal_steel_policy = load_selected_metal_steel_policy()
    from por_supplier_linking import configure_por_export_support, attach_por_supplier_linking
    from por_scenario_inputs import fresh_scenario_definitions
    configure_por_export_support()
    indices = list(range(len(scenarios))) if BRIGHTWAY_SCENARIO_INDICES is None else list(BRIGHTWAY_SCENARIO_INDICES)
    if not indices or len(set(indices)) != len(indices) or any(i < 0 or i >= len(scenarios) for i in indices):
        raise ValueError("Select unique, valid Brightway scenario indices.")
    bw_scenarios = fresh_scenario_definitions([
        dict(scenarios[i], year=year) for year in BRIGHTWAY_YEARS for i in indices
    ])
    bw_scenario_names = [
        scenario_names[i].replace(f"_{scenarios[i]['year']}_", f"_{year}_", 1)
        + selected_metal_steel_policy.case_suffix
        for year in BRIGHTWAY_YEARS for i in indices
    ]

    #pr.clear_cache() 

    # Both export routes must use the same workbook staged in setup.
    if itom_inventory_snapshot is not None:
        verify_itom_inventory_snapshot(itom_inventory_snapshot)
    # Build successfully before replacing the existing scenario databases.
    ndb = pr.NewDatabase(
        scenarios = bw_scenarios,
        source_db=DB_NAME_INIT, # <-- name of the database in the BW2 project. Must be a string.
        source_version=EI_VERSION, # <-- version of ecoinvent. Must be a string.
        key=KEY_PREMISE, # <-- ask the key to run premise from the developers
        biosphere_name=BIOSPHERE_DB)

    # Policy runs after native updates, before PoR links and export validation.
    metal_policy_module.attach_metal_steel_policy(ndb, policy=selected_metal_steel_policy, stage="brightway")
    attach_por_supplier_linking(ndb, stage="brightway")
    # Carbon balance and scenario-specific feedstock CO2 origin, after supplier linking.
    if ASSESSMENT_SCOPE == "por_polymers":
        import por_carbon_emissions as carbon_emissions
        importlib.reload(carbon_emissions)
        carbon_emissions.attach_por_carbon_emissions(ndb, stage="brightway")
    ndb.update()
    display(pd.DataFrame([{k: r.get(k) for k in ("case", "year", "changed_activities", "balance_reviews", "report_path")}
                          for r in getattr(ndb, "por_carbon_emission_reports", [])]))
    metal_audit_fields = ("year", "external_scenarios", "nonferrous_case", "steel_owner", "changed_markets", "warning_rows", "report_path")
    display(pd.DataFrame([{key: report.get(key) for key in metal_audit_fields}
                          for report in getattr(ndb, "metal_steel_policy_reports", [])]))

    # Check checkpoint readability before any existing target is removed.
    bw_export_preflight = por_linking.preflight_por_brightway_export(ndb)
    print(f"Brightway inventory preflight passed for {len(bw_export_preflight)} scenario(s).", flush=True)
    # Replace only these export targets, preserving unrelated future databases.
    for db in bw_scenario_names:
        if db in bd.databases:
            print(f"Replacing {db} after successful scenario update and inventory preflight")
            del bd.databases[db]

    ndb.write_db_to_brightway(name=bw_scenario_names)

Metals share data: reused ; products: 104
premise v.(2, 5, 5)
+---------------------------------------------------------------------+
| Warning                                                             |
+---------------------------------------------------------------------+
| Some scenarios include negative emission technologies, such as      |
| carbon dioxide removal (CDR). We recommend using a climate          |
| change indicator that accounts for both the uptake and release      |
| of non-fossil CO2, for example:                                     |
|                                                                     |
| IPCC 2021 (incl. biogenic CO2)                                      |
|   climate change: total (incl. biogenic CO2)                        |
|   global warming potential (GWP100)                                 |
|                                                                     |
| Alternatively, use the methods installed by `premise_gwp`,          |
| 

Processing scenarios for all sectors:   0%|     | 0/2 [00:00<?, ?it/s]

Importing C:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\inventories/lci-forecast.xlsx
Extracted 1 worksheets in 0.01 seconds
Migration route: 3.10 → 3.11 → 3.12
Applying forward migration 3.10 -> 3.11
Applying forward migration 3.11 -> 3.12
Importing C:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\inventories/lci-itom_por.xlsx
Extracted 1 worksheets in 0.63 seconds
  Native metals audit: scanning activity metadata (no background exchanges)
  Native metals audit: planning 10 selected identity record(s)


Processing scenarios for all sectors:  50%|▌| 1/2 [01:28<01:28, 88.68s

Importing C:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\inventories/lci-forecast.xlsx
Extracted 1 worksheets in 0.01 seconds
Migration route: 3.10 → 3.11 → 3.12
Applying forward migration 3.10 -> 3.11
Applying forward migration 3.11 -> 3.12
Importing C:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\inventories/lci-itom_por.xlsx
Extracted 1 worksheets in 0.10 seconds
  Native metals audit: scanning activity metadata (no background exchanges)
  Native metals audit: planning 10 selected identity record(s)


Processing scenarios for all sectors: 100%|█| 2/2 [02:56<00:00, 88.15s


... (2 outputs left out)

,year,external_scenarios,nonferrous_case,steel_owner,changed_markets,warning_rows,report_path
0,2050,"[fc_nz, OCE]",baseline,endogenous,0,0,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
1,2050,"[fc_nz, CL]",baseline,endogenous,0,0,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...


Brightway inventory preflight passed for 2 scenario(s).
Replacing remind_2050_SSP2-PkBudg1000_fc_nz_OCE after successful scenario update and inventory preflight
11:56:19+0100 [info     ] Vacuuming database            
Replacing remind_2050_SSP2-PkBudg1000_fc_nz_CL after successful scenario update and inventory preflight
11:59:51+0100 [info     ] Vacuuming database            
Write new database(s) to Brightway.


Brightway database written: remind_2050_SSP2-PkBudg1000_fc_nz_OCE


Brightway database written: remind_2050_SSP2-PkBudg1000_fc_nz_CL
Generate scenario report.
Report saved under c:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\export\scenario_report.
Generate change report.


### 2. Export the defined scenarios for chosen years as data packages needed for PATHWAYS in the next step (see `2_calc_impacts.ipynb`)

Classification metadata is captured after updating/linking, **before** premise releases its inventories during matrix export. The package receives a populated CPC/ISIC classification table; detailed assignment sources, provisional rules and conflicts are saved under `export/por_classifications/<snapshot hash>/`. Existing native labels take priority. Explicit foreground rules fill justified gaps; unknowns remain `undefined`, not omitted from impact totals. This changes reporting metadata only, not exchanges or LCIA factors.

Restart the kernel and use a fresh builder after helper changes. See [classification workflow](docs/por_classifications.md) and [rule sources](docs/por_classification_rules_sources.md).

In [13]:
if create_db_package:
    import importlib, sys
    from config import store_generated_package
    import premise_compat as por_compat
    import por_classifications as por_classification_details
    import inventory_supplier_compat as por_inventory_compat
    import por_supplier_linking as por_linking
    import por_scenario_inputs as por_inputs
    import metals_scenario_policy as metal_policy_module
    import primary_vs_secondary_metals.share_files as metal_share_files
    # Ordinary imports keep old helper functions in a long-lived kernel.
    importlib.reload(por_compat)
    importlib.reload(por_classification_details)
    importlib.reload(por_inventory_compat)
    importlib.reload(por_linking)
    importlib.reload(por_inputs)
    importlib.reload(metal_policy_module)
    importlib.reload(metal_share_files)
    selected_metal_steel_policy = load_selected_metal_steel_policy()
    selected_package_name = PATHWAYS_PACKAGE_BASENAME + selected_metal_steel_policy.case_suffix

    print(f"Running premise {getattr(pr, '__version__', 'unknown')} from {getattr(pr, '__file__', 'unknown')}; Python: {sys.executable}", flush=True)

    por_linking.configure_por_export_support()
    package_scenarios = por_inputs.fresh_scenario_definitions(scenarios)
    print("Preparing fresh inputs for the multiple Pathways package...", flush=True)

    if itom_inventory_snapshot is not None:
        verify_itom_inventory_snapshot(itom_inventory_snapshot)
    #if 'external scenarios' in str(scenario):
    #pr.clear_cache()
    ndb = pr.PathwaysDataPackage(
        scenarios=package_scenarios,
        years=PACKAGE_YEARS,
        source_db=DB_NAME_INIT,
        source_version=EI_VERSION,
        key=KEY_PREMISE,
        biosphere_name=BIOSPHERE_DB )

    # The package calls its internal NewDatabase.update() before exporting matrices.
    print("Package builder ready; attaching the PoR post-update linking step.", flush=True)
    metal_policy_module.attach_metal_steel_policy(ndb.datapackage, policy=selected_metal_steel_policy, stage="pathways")
    metal_policy_module.attach_metal_steel_package_provenance(ndb, policy=selected_metal_steel_policy)
    if ASSESSMENT_SCOPE == "por_polymers":
        import por_carbon_emissions as carbon_emissions
        importlib.reload(carbon_emissions)
        # Order matters: native updates -> supplier links -> CO2-origin split -> mapping/classifications -> export.
        por_linking.attach_por_supplier_linking(ndb.datapackage, stage="pathways")
        carbon_emissions.attach_por_carbon_emissions(ndb.datapackage, stage="pathways")
        carbon_emissions.attach_carbon_package_provenance(ndb)
        
    por_linking.attach_por_pathways_export(ndb, stage="pathways")
    
    ndb.create_datapackage(
        name=selected_package_name,
        contributors=[
            {"name": "Tom Terlouw",
            "email": "tom.terlouw@psi.ch"}] )

    selected_package_path = store_generated_package(selected_package_name)
    print("Selected archive for notebook 2:", str(selected_package_path))
    metal_audit_fields = ("year", "external_scenarios", "nonferrous_case", "steel_owner", "changed_markets", "warning_rows", "report_path")
    display(pd.DataFrame([{key: report.get(key) for key in metal_audit_fields}
                          for report in getattr(ndb.datapackage, "metal_steel_policy_reports", [])]))
    display(pd.DataFrame([{k: r.get(k) for k in ("case", "year", "changed_activities", "balance_reviews", "report_path")}
                          for r in getattr(ndb.datapackage, "por_carbon_emission_reports", [])]))
    # Exported classification metadata comes from the pre-matrix snapshot.
    print("Classification export summary:", ndb.classification_stats)
    print("Classification audit directory:", getattr(ndb, "classification_audit_path", None))
    display(pd.DataFrame(ndb.classification_audit).head(30))

Metals share data: reused ; products: 104
Running premise (2, 5, 5) from c:\Users\terlouw_t\.conda\envs\lca_transience_255\Lib\site-packages\premise\__init__.py; Python: c:\Users\terlouw_t\.conda\envs\lca_transience_255\python.exe
Preparing fresh inputs for the multiple Pathways package...
premise v.(2, 5, 5)
+---------------------------------------------------------------------+
| Warning                                                             |
+---------------------------------------------------------------------+
| Some scenarios include negative emission technologies, such as      |
| carbon dioxide removal (CDR). We recommend using a climate          |
| change indicator that accounts for both the uptake and release      |
| of non-fossil CO2, for example:                                     |
|                                                                     |
| IPCC 2021 (incl. biogenic CO2)                                      |
|   climate change: total (incl. biogenic

Processing scenarios for all sectors:   0%|    | 0/12 [00:00<?, ?it/s]

Importing C:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\inventories/lci-forecast.xlsx
Extracted 1 worksheets in 0.02 seconds
Migration route: 3.10 → 3.11 → 3.12
Applying forward migration 3.10 -> 3.11
Applying forward migration 3.11 -> 3.12
Importing C:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\inventories/lci-itom_por.xlsx
Extracted 1 worksheets in 0.13 seconds
No suppliers found for market for toluene (por) in NL. No market created. This may cause linking issue.
No suppliers found for market for methanol (por) in NL. No market created. This may cause linking issue.
No suppliers found for market for syngas, CO component (por) in NL. No market created. This may cause linking issue.
No suppliers found for market for syngas, CO2 component (por) in NL. No market created. This may cause linking issue.
  Native metals audit: scanning activity metadata (no background exchanges)
  Native metals audit: planning 14 selected identity record(s)


Processing scenarios for all sectors:   8%| | 1/12 [02:11<24:04, 131.3

Importing C:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\inventories/lci-forecast.xlsx
Extracted 1 worksheets in 0.01 seconds
Migration route: 3.10 → 3.11 → 3.12
Applying forward migration 3.10 -> 3.11
Applying forward migration 3.11 -> 3.12
Importing C:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\inventories/lci-itom_por.xlsx
Extracted 1 worksheets in 0.11 seconds
No suppliers found for market for toluene (por) in NL. No market created. This may cause linking issue.
No suppliers found for market for methanol (por) in NL. No market created. This may cause linking issue.
  Native metals audit: scanning activity metadata (no background exchanges)
  Native metals audit: planning 14 selected identity record(s)


Processing scenarios for all sectors:  17%|▏| 2/12 [04:13<21:01, 126.1

... (21 outputs left out)

,year,external_scenarios,nonferrous_case,steel_owner,changed_markets,warning_rows,report_path
0,2025,"[fc_nz, OCE]",baseline,endogenous,0,0,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
1,2025,"[fc_nz, CL]",baseline,endogenous,0,0,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
2,2030,"[fc_nz, OCE]",baseline,endogenous,0,0,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
3,2030,"[fc_nz, CL]",baseline,endogenous,0,0,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
4,2035,"[fc_nz, OCE]",baseline,endogenous,0,0,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
5,2035,"[fc_nz, CL]",baseline,endogenous,0,0,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
6,2040,"[fc_nz, OCE]",baseline,endogenous,0,0,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
7,2040,"[fc_nz, CL]",baseline,endogenous,0,0,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
8,2045,"[fc_nz, OCE]",baseline,endogenous,0,0,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
9,2045,"[fc_nz, CL]",baseline,endogenous,0,0,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...


,case,year,changed_activities,balance_reviews,report_path
0,OCE,2025,0,0,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
1,CL,2025,0,0,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
2,OCE,2030,0,1,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
3,CL,2030,0,1,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
4,OCE,2035,0,1,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
5,CL,2035,0,1,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
6,OCE,2040,0,1,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
7,CL,2040,0,1,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
8,OCE,2045,8,1,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...
9,CL,2045,0,1,C:\Users\terlouw_t\Documents\Projects\TRANSIEN...


Classification export summary: {'activity_count': 483118, 'unique_pairs': 13900, 'classification_rows': 32270, 'undefined_pairs': {'CPC': 122, 'ISIC rev.4 ecoinvent': 241}, 'status_counts': {'original_metadata': 31667, 'provisional': 158, 'undefined': 350, 'conflicting_metadata': 22, 'normalized_reference': 19, 'exact_reference': 54}, 'conflicts': 22, 'ambiguous_reference_entries': 41, 'scenario_count': 12}
Classification audit directory: c:\Users\terlouw_t\Documents\Projects\TRANSIENCE\lca_transience\export\por_classifications\253dce936abfa5c6


,name,reference product,classification_system,classification_code,status,source,rule_id,reference_name,reference_product,reason,units,locations,scenarios,activity_count
0,"(CF3SO2)2NH production, for LiS battery",(CF3SO2)2NH,CPC,46410: Primary cells and primary batteries,original_metadata,scenario_metadata,,"(CF3SO2)2NH production, for LiS battery",(CF3SO2)2NH,Preserved dataset classification metadata.,kilogram,GLO,remind/SSP2-PkBudg1000/2025/0; remind/SSP2-PkB...,12
1,"(CF3SO2)2NH production, for LiS battery",(CF3SO2)2NH,ISIC rev.4 ecoinvent,2720:Manufacture of batteries and accumulators,original_metadata,scenario_metadata,,"(CF3SO2)2NH production, for LiS battery",(CF3SO2)2NH,Preserved dataset classification metadata.,kilogram,GLO,remind/SSP2-PkBudg1000/2025/0; remind/SSP2-PkB...,12
2,"(CF3SO2)2NNa production, for LiS battery",(CF3SO2)2NNa,CPC,46410: Primary cells and primary batteries,original_metadata,scenario_metadata,,"(CF3SO2)2NNa production, for LiS battery",(CF3SO2)2NNa,Preserved dataset classification metadata.,kilogram,GLO,remind/SSP2-PkBudg1000/2025/0; remind/SSP2-PkB...,12
3,"(CF3SO2)2NNa production, for LiS battery",(CF3SO2)2NNa,ISIC rev.4 ecoinvent,2720:Manufacture of batteries and accumulators,original_metadata,scenario_metadata,,"(CF3SO2)2NNa production, for LiS battery",(CF3SO2)2NNa,Preserved dataset classification metadata.,kilogram,GLO,remind/SSP2-PkBudg1000/2025/0; remind/SSP2-PkB...,12
4,"(Me3Si)2NH production, for LiS battery",(Me3Si)2NH,CPC,46410: Primary cells and primary batteries,original_metadata,scenario_metadata,,"(Me3Si)2NH production, for LiS battery",(Me3Si)2NH,Preserved dataset classification metadata.,kilogram,GLO,remind/SSP2-PkBudg1000/2025/0; remind/SSP2-PkB...,12
5,"(Me3Si)2NH production, for LiS battery",(Me3Si)2NH,ISIC rev.4 ecoinvent,2720:Manufacture of batteries and accumulators,original_metadata,scenario_metadata,,"(Me3Si)2NH production, for LiS battery",(Me3Si)2NH,Preserved dataset classification metadata.,kilogram,GLO,remind/SSP2-PkBudg1000/2025/0; remind/SSP2-PkB...,12
6,"1,1,1-trichloroethane production, vinyl chlori...","1,1,1-trichloroethane",CPC,"3411: Hydrocarbons and their halogenated, sulp...",original_metadata,scenario_metadata,,"1,1,1-trichloroethane production, vinyl chlori...","1,1,1-trichloroethane",Preserved dataset classification metadata.,kilogram,GLO,remind/SSP2-PkBudg1000/2025/0; remind/SSP2-PkB...,12
7,"1,1,1-trichloroethane production, vinyl chlori...","1,1,1-trichloroethane",ISIC rev.4 ecoinvent,2011:Manufacture of basic chemicals,original_metadata,scenario_metadata,,"1,1,1-trichloroethane production, vinyl chlori...","1,1,1-trichloroethane",Preserved dataset classification metadata.,kilogram,GLO,remind/SSP2-PkBudg1000/2025/0; remind/SSP2-PkB...,12
8,"1,1,2-trichloroethane production, vinyl chlori...","1,1,2-trichloroethane",CPC,"3411: Hydrocarbons and their halogenated, sulp...",original_metadata,scenario_metadata,,"1,1,2-trichloroethane production, vinyl chlori...","1,1,2-trichloroethane",Preserved dataset classification metadata.,kilogram,GLO,remind/SSP2-PkBudg1000/2025/0; remind/SSP2-PkB...,12
9,"1,1,2-trichloroethane production, vinyl chlori...","1,1,2-trichloroethane",ISIC rev.4 ecoinvent,2011:Manufacture of basic chemicals,original_metadata,scenario_metadata,,"1,1,2-trichloroethane production, vinyl chlori...","1,1,2-trichloroethane",Preserved dataset classification metadata.,kilogram,GLO,remind/SSP2-PkBudg1000/2025/0; remind/SSP2-PkB...,12


### 3. Continue in Notebook `2_calc_impacts.ipynb` to calculate environmental burdens of sectors or entire energy system scenarios using the exported data packages from the previous step

## Optional additional carbon-accounting preflight

This is a separate workflow after package export. It does **not** alter the package, inventories, markets, supplier linking, carbon flows or LCIA methods. The shared module validates the companion assumptions and records source identities. The full production-carbon, lifetime/end-of-life and correction-review analyses are at the end of notebook 2.

`configuration_file/carbon_workflows.yaml` connects the source ITOM workbooks to the CSV tables under `carbon_workflows/`. Raw workbook volumes are per five-year period and are divided by five; the converted IAMC CSV is already annual. Application lifetimes and waste destinations are explicitly labelled screening cases until measured data are available. The engine follows carbon in kg C and preserves unknown composition/origin. MDI and polyols remain in gate accounts but are excluded from polymer in-use stocks until their conversion to final articles is defined.

Enable the check below when preparing a carbon-accounting run. It only reads inputs. See [the additional workflows and proposed corrections](docs/carbon_workflows.md).


In [14]:
RUN_ADDITIONAL_CARBON_PREFLIGHT = False

if RUN_ADDITIONAL_CARBON_PREFLIGHT:
    import importlib
    import carbon_accounting
    carbon_accounting = importlib.reload(carbon_accounting)
    carbon_preflight = carbon_accounting.carbon_workflow_preflight()
    display(carbon_preflight["sources"])
    print("Connected carbon assumptions validated. No inventory or package corrections applied.")
else:
    print("Optional carbon preflight disabled. Existing package workflow is unchanged.")


Optional carbon preflight disabled. Existing package workflow is unchanged.
